**Install Libraries**

In [ ]:
pip install  -U langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [ ]:
pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.4 MB/s eta 0:00:00


**Setting Up LLM using API Key**

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] =  userdata.get("GEMINI_API_KEY")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
      temperature=0,timeout=30,max_retries=2)



**Import Required modules from Langchain**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts.chat import SystemMessagePromptTemplate,HumanMessagePromptTemplate
from langchain_core.prompts import PromptTemplate



**Create Persona**

In [ ]:
template = """

You are a travel planner agent who answers only travel related questions like flights, hotels, destinations and travel trips.

The user will input the destination {destination} and number of days  of vacation {num_of_days}



If a user asks non travel related questions then repsond with prompts

"I can't help with it!!"


"""


In [ ]:
##Message List

messages = [
            {"role":"system","content":template},
            {"role":"human","content": "Plan a vacation to {destination} for {num_of_days}"}
    ]
prompt_template = PromptTemplate.from_template(template)

In [ ]:
#destination_input = input("Enter a Destination:::")

In [ ]:
#vacation_days = input("Enter nummber of days:::")

In [ ]:
#formatted_prompt = prompt_template.format_prompt(destination = destination_input,num_of_days = vacation_days)

In [ ]:
#response = llm.invoke(formatted_prompt)
#messages.append({"role":"assistant","content": response.content})
#rint(response)

**Function to manage memory and summarize**

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

chat_history = []

prompt = ChatPromptTemplate.from_messages([

("system",template),
MessagesPlaceholder(variable_name= "chat_history"),
("human", "{destination},{num_of_days}")
])

chain = prompt | llm

def summarize_history(history):
  summary_prompt = f"Summarize the following:"+ {history[:-2]}
  summary = llm.invoke(summary_prompt).content
  return [SystemMessage(content=f"Previous summary: {summary}")] + history[-2:]


def chat_function(dest, no_days):
  global chat_history


  if len(chat_history) > 5:
        chat_history = summarize_history(chat_history)

  result = chain.invoke({"destination":dest,"num_of_days":no_days,"chat_history":chat_history})
  chat_history.append(HumanMessage(content=f"Destination: {dest}, Number of Days: {no_days}"))
  chat_history.append(AIMessage(content=result.content))
  return result.content

**Call The Chain Method**

In [ ]:
print(chat_function("Paris",5))
print("********************************************************")

print(chat_function("Munich",5))
print("********************************************************")

print(chat_function("New York",5))
print("********************************************************")


print(chat_function("San Francisco",7))
print(chat_function("Las Vegas",6))
print(chat_function("Salt Lake City",6))


Great! Paris for 5 days sounds wonderful.

To help you plan, what kind of information are you looking for? For example, would you like:

*   **Flight and hotel suggestions?**
*   **Itinerary ideas for 5 days in Paris?**
*   **Recommendations for attractions, restaurants, or activities?**
*   **Tips for getting around?**
********************************************************
Excellent! Munich for 5 days is a fantastic choice.

To help you plan your trip, what kind of information would you like to start with? For instance, are you interested in:

*   **Flight and hotel recommendations?**
*   **Itinerary suggestions for 5 days in Munich?**
*   **Ideas for attractions, food, or day trips?**
*   **Tips for transportation within the city?**
********************************************************


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 24.290531319s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
chat_history

[HumanMessage(content='Destination: Paris, Number of Days: 5', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Great choice! Paris for 5 days sounds wonderful.\n\nTo help you plan your trip, could you tell me what kind of experience you're looking for? For example:\n\n*   Are you interested in **sightseeing and famous landmarks** (Eiffel Tower, Louvre, Notre Dame)?\n*   Do you prefer **art and culture** (museums, galleries, theatre)?\n*   Are you looking for **food and culinary experiences** (restaurants, markets, cooking classes)?\n*   Do you enjoy **shopping** or exploring **local neighborhoods**?\n*   Are you traveling **solo, as a couple, or with family**?\n\nOnce I have a better idea of your interests, I can suggest some itineraries, hotels, and activities!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Destination: How does universe works, Number of Days: 0', additional_kwargs={}, response_metadata={}),

**Stream the response**